In [ ]:
import onnxruntime as ort
import numpy as np
import torch
from torchvision import transforms
from PIL import Image

# 1. ONNX 런타임 세션 로드
model_path = "mission_16_resnet18.onnx"
ort_session = ort.InferenceSession(model_path)

# 2. 테스트 데이터셋 준비 (예: 이미지 로드 및 전처리)
# 기존 미션에서 사용했던 Transform과 동일해야 합니다.
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 이미지 불러오기 (테스트용 이미지 경로)
img = Image.open("./test_image.jpg")
input_tensor = transform(img).unsqueeze(0) # (1, 3, 224, 224)

# 3. ONNX 추론 실행
# ONNX 런타임은 numpy array를 입력으로 받습니다.
def to_numpy(tensor):
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

ort_inputs = {ort_session.get_inputs()[0].name: to_numpy(input_tensor)}
ort_outs = ort_session.run(None, ort_inputs)

# 4. 결과 확인
print("Inference Output Shape:", ort_outs[0].shape)
predicted_idx = np.argmax(ort_outs[0])
print(f"Predicted Class Index: {predicted_idx}")